In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

PINECONE_API_KEY = os.getenv("PINE_CONE_API_KEY")


In [2]:
from pinecone import Pinecone 
pc = Pinecone(api_key=PINECONE_API_KEY)
pc


In [3]:
from langchain_openai import OpenAIEmbeddings
embeddings = OpenAIEmbeddings(
    model = "text-embedding-3-small",
    dimensions=1024,
    api_key = os.getenv("OPENAI_API_KEY")
)
embeddings

OpenAIEmbeddings(client=<openai.resources.embeddings.Embeddings object at 0x00000144446F6E40>, async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x000001445CF794F0>, model='text-embedding-3-small', dimensions=1024, deployment='text-embedding-ada-002', openai_api_version=None, openai_api_base=None, openai_api_type=None, openai_proxy=None, embedding_ctx_length=8191, openai_api_key=SecretStr('**********'), openai_organization=None, allowed_special=None, disallowed_special=None, chunk_size=1000, max_retries=2, request_timeout=None, headers=None, tiktoken_enabled=True, tiktoken_model_name=None, show_progress_bar=False, model_kwargs={}, skip_empty=False, default_headers=None, default_query=None, retry_min_seconds=4, retry_max_seconds=20, http_client=None, http_async_client=None, check_embedding_ctx_length=True)

In [4]:
# Connect to the PineCone DB

from pinecone import ServerlessSpec

index_name = "rag-poc-index"   

if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=1024,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )

index = pc.Index(index_name)
index

In [5]:
from langchain_pinecone import PineconeVectorStore

vector_store =PineconeVectorStore(
    index =index,
    embedding = embeddings
)

vector_store

In [6]:
from langchain_core.documents import Document

document_1 = Document(
    page_content="I had chocolate chip pancakes and scrambled eggs for breakfast this morning.",
    metadata={"source": "tweet"},
)

document_2 = Document(
    page_content="The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.",
    metadata={"source": "news"},
)

document_3 = Document(
    page_content="Building an exciting new project with LangChain - come check it out!",
    metadata={"source": "tweet"},
)

document_4 = Document(
    page_content="Robbers broke into the city bank and stole $1 million in cash.",
    metadata={"source": "news"},
)

document_5 = Document(
    page_content="Wow! That was an amazing movie. I can't wait to see it again.",
    metadata={"source": "tweet"},
)

document_6 = Document(
    page_content="Is the new iPhone worth the price? Read this review to find out.",
    metadata={"source": "website"},
)

document_7 = Document(
    page_content="The top 10 soccer players in the world right now.",
    metadata={"source": "website"},
)

document_8 = Document(
    page_content="LangGraph is the best framework for building stateful, agentic applications!",
    metadata={"source": "tweet"},
)

document_9 = Document(
    page_content="The stock market is down 500 points today due to fears of a recession.",
    metadata={"source": "news"},
)

document_10 = Document(
    page_content="I have a bad feeling I am going to get deleted :(",
    metadata={"source": "tweet"},
)

documents = [
    document_1,
    document_2,
    document_3,
    document_4,
    document_5,
    document_6,
    document_7,
    document_8,
    document_9,
    document_10,
]
documents

[Document(metadata={'source': 'tweet'}, page_content='I had chocolate chip pancakes and scrambled eggs for breakfast this morning.'),
 Document(metadata={'source': 'news'}, page_content='The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.'),
 Document(metadata={'source': 'tweet'}, page_content='Building an exciting new project with LangChain - come check it out!'),
 Document(metadata={'source': 'news'}, page_content='Robbers broke into the city bank and stole $1 million in cash.'),
 Document(metadata={'source': 'tweet'}, page_content="Wow! That was an amazing movie. I can't wait to see it again."),
 Document(metadata={'source': 'website'}, page_content='Is the new iPhone worth the price? Read this review to find out.'),
 Document(metadata={'source': 'website'}, page_content='The top 10 soccer players in the world right now.'),
 Document(metadata={'source': 'tweet'}, page_content='LangGraph is the best framework for building stateful, agentic application

In [7]:
# vector_store.add_documents(documents = documents)
vector_store.add_documents(documents=documents)

['16658ccd-fab1-4086-a60e-77e8fe208a72',
 '66ce67a4-a7fb-44d5-b6c0-a0f6e6f3e383',
 'ec602381-7a09-40f7-8595-14bd2af87685',
 'f82cb45f-2a78-4136-91ba-391f0a21de86',
 '86979ffc-6521-43ed-adb0-a38a283a68cd',
 '7813c9fa-ee9d-4502-8ff9-dfcc39338221',
 'a6e8ab5f-faae-4f5f-b065-d9cec890bdb3',
 '0763f8fe-c950-4d63-afe4-6c7c9640422d',
 '154328cf-3d0a-48b0-be72-f58f5fdf6f53',
 '74a362a6-a292-4d24-9580-e5ad128cea70']

HTTP response headers: HTTPHeaderDict({'Date': 'Tue, 06 Jan 2026 14:28:03 GMT', 'Content-Type': 'application/json', 'Content-Length': '104', 'Connection': 'keep-alive', 'x-pinecone-request-latency-ms': '515', 'x-pinecone-request-id': '4313664714456087271', 'x-envoy-upstream-service-time': '58', 'x-pinecone-response-duration-ms': '518', 'server': 'envoy'})
HTTP response body: {"code":3,"message":"Vector dimension 1536 does not match the dimension of the index 1024","details":[]}

Recreted the index with dimention and embedding with 1024

In [9]:
## Qurey the Index

results = vector_store.similarity_search(
    "LangChain provides abstractions to make working with LLMs easy",
    k=2,
    filter ={"source":"tweet"}
)

for res in results:
    print(f" * {res.page_content} [{res.metadata}]")

 * Building an exciting new project with LangChain - come check it out! [{'source': 'tweet'}]
 * LangGraph is the best framework for building stateful, agentic applications! [{'source': 'tweet'}]


In [10]:
results = vector_store.similarity_search_with_score(
    "Will it be hot tomorrow?",
    k=1,
    filter ={"source":"news"}
)

for res, score in results:
     print(f" * {res.page_content} [{res.metadata}]")

 * The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees. [{'source': 'news'}]


In [ ]:
# Retriever 

retriever = vector_store.as_retriever(
    search_type ="similarity_score_threshold",
    search_kwargs={"k":1,"score_threshold":0.4},
)

retriever.invoke("Stealing from the bank is Crime ", filter={"source":"news"})

[Document(id='f82cb45f-2a78-4136-91ba-391f0a21de86', metadata={'source': 'news'}, page_content='Robbers broke into the city bank and stole $1 million in cash.')]

: 